# KNRM, ColBERT & Sentence-BERT Encoder Tests

In [ ]:
import sys
sys.path.insert(0, '/Users/skyler/Projects/document_retrieval_project')

import time
import numpy as np
import torch
import faiss
import matplotlib.pyplot as plt

from loader import load_data
from encoders.knrm import KNRMScorer
from encoders.colbert import ColBERTScorer
from encoders.sbert import SentenceBERTEncoder

In [ ]:
# Load a small sample of MS MARCO passages + queries with known relevant passages
ds = load_data()
passages_text = []
queries_with_answers = []  # list of (query_str, [relevant_global_indices])

for example in ds:
    start = len(passages_text)
    for passage in example['passages']['passage_text']:
        passages_text.append(passage)
    relevant = [
        start + i
        for i, label in enumerate(example['passages']['is_selected'])
        if label == 1
    ]
    if relevant:
        queries_with_answers.append((example['query'], relevant))
    if len(passages_text) >= 100:
        break

print(f"Passages: {len(passages_text)}")
print(f"Queries with relevant passages: {len(queries_with_answers)}")

## Dense Retrieval with Sentence-BERT (FAISS)

In [ ]:
sbert = SentenceBERTEncoder(batch_size=64)

t0 = time.perf_counter()
sbert_embs = sbert.encode(passages_text).astype(np.float32)
sbert_index_time = time.perf_counter() - t0

# Normalize and build FAISS index
norms = np.linalg.norm(sbert_embs, axis=1, keepdims=True)
sbert_embs_norm = sbert_embs / norms
index_sbert = faiss.IndexFlatIP(sbert_embs_norm.shape[1])
index_sbert.add(sbert_embs_norm)

print(f"SBERT index built in {sbert_index_time:.2f}s — {sbert_embs_norm.shape}")

In [ ]:
def sbert_retrieve(query: str, k: int = 10):
    q_emb = sbert.encode([query]).astype(np.float32)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    _, indices = index_sbert.search(q_emb, k)
    return indices[0].tolist()

# Smoke test
q, rel = queries_with_answers[0]
results = sbert_retrieve(q)
print(f"Query: {q}")
print(f"Relevant: {rel}")
print(f"Top-10 SBERT: {results}")
print(f"Hit: {any(i in rel for i in results)}")

## Dense Retrieval with KNRM

In [ ]:
knrm = KNRMScorer()
print(f"KNRM device: {knrm.device}, kernels: {knrm.ranker.n_kernels}")

def knrm_retrieve(query: str, passages: list[str], k: int = 10):
    scores = knrm.score_pairs(query, passages)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked[:k], scores

# Smoke test
q, rel = queries_with_answers[0]
top_k, scores = knrm_retrieve(q, passages_text)
print(f"\nQuery: {q}")
print(f"Relevant: {rel}")
print(f"Top-10 KNRM: {top_k}")
print(f"Hit: {any(i in rel for i in top_k)}")

## Dense Retrieval with ColBERT

In [ ]:
colbert = ColBERTScorer()
print(f"ColBERT device: {colbert.device}, projection dim: {colbert.encoder.dim}")

def colbert_retrieve(query: str, passages: list[str], k: int = 10):
    scores = colbert.score_pairs(query, passages)
    ranked = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    return ranked[:k], scores

# Smoke test
q, rel = queries_with_answers[0]
top_k, scores = colbert_retrieve(q, passages_text)
print(f"\nQuery: {q}")
print(f"Relevant: {rel}")
print(f"Top-10 ColBERT: {top_k}")
print(f"Hit: {any(i in rel for i in top_k)}")

## MRR@10 and Latency Comparison

In [ ]:
def mrr_at_k(ranked_indices: list[int], relevant: list[int], k: int = 10) -> float:
    for rank, idx in enumerate(ranked_indices[:k], 1):
        if idx in relevant:
            return 1.0 / rank
    return 0.0

results = {"SBERT": {"mrr": [], "latency": []},
           "KNRM":  {"mrr": [], "latency": []},
           "ColBERT": {"mrr": [], "latency": []}}

for query, relevant in queries_with_answers:
    # SBERT
    t0 = time.perf_counter()
    sbert_top = sbert_retrieve(query, k=10)
    results["SBERT"]["latency"].append(time.perf_counter() - t0)
    results["SBERT"]["mrr"].append(mrr_at_k(sbert_top, relevant))

    # KNRM
    t0 = time.perf_counter()
    knrm_top, _ = knrm_retrieve(query, passages_text, k=10)
    results["KNRM"]["latency"].append(time.perf_counter() - t0)
    results["KNRM"]["mrr"].append(mrr_at_k(knrm_top, relevant))

    # ColBERT
    t0 = time.perf_counter()
    cb_top, _ = colbert_retrieve(query, passages_text, k=10)
    results["ColBERT"]["latency"].append(time.perf_counter() - t0)
    results["ColBERT"]["mrr"].append(mrr_at_k(cb_top, relevant))

print(f"{'Model':<10} {'MRR@10':>8} {'Avg latency (s)':>16}")
print("-" * 36)
for model, data in results.items():
    mrr = np.mean(data["mrr"])
    lat = np.mean(data["latency"])
    print(f"{model:<10} {mrr:>8.4f} {lat:>16.4f}")

In [ ]:
models = list(results.keys())
mrr_vals = [np.mean(results[m]["mrr"]) for m in models]
lat_vals = [np.mean(results[m]["latency"]) for m in models]
colors = ["steelblue", "seagreen", "darkorange"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.bar(models, mrr_vals, color=colors)
ax1.set_title("MRR@10")
ax1.set_ylabel("MRR@10")
ax1.set_ylim(0, 1)
for i, v in enumerate(mrr_vals):
    ax1.text(i, v + 0.01, f"{v:.3f}", ha="center", fontsize=10)

ax2.bar(models, lat_vals, color=colors)
ax2.set_title("Avg Query Latency (s)")
ax2.set_ylabel("Seconds")
for i, v in enumerate(lat_vals):
    ax2.text(i, v + 0.001, f"{v:.3f}s", ha="center", fontsize=10)

plt.suptitle("SBERT vs KNRM vs ColBERT — MRR@10 and Latency")
plt.tight_layout()
plt.show()